<a href="https://colab.research.google.com/github/cyrus2281/notes/blob/main/Architecture/Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Software Architecture Notes

>[Software Architecture Notes](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=PQ6Eg6AkHf12)

>[Event-Driven Architecture](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>[Request/Reply Pattern](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>[Core Mechanism](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>[Implementation Techniques](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>>[Correlation IDs](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>>>[Temporary Queues](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=TG1TXEEiVZEz)

>>[How Apache Kafka Differs From Standard Messaging](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=pS5Q0MhMaH5z)

>>>[Apache Kafka (Streaming Platform)](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=pS5Q0MhMaH5z)

>>>[Standard Messaging (RabbitMQ, ActiveMQ, JMS)](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=pS5Q0MhMaH5z)

>>>[Key Comparison Summary](#updateTitle=true&folderId=1a_opiJpdQAo6ihcK8nHLrZT2VNhX7c9h&scrollTo=pS5Q0MhMaH5z)



# Event-Driven Architecture

## Request/Reply Pattern

The **Request/Reply pattern** allows for synchronous-like behavior ("pseudo-synchronous messaging") within an asynchronous event-driven architecture.

### Core Mechanism
*   **Structure:** Utilizes two distinct queues—a **Request Queue** (for sending) and a **Reply Queue** (for receiving).
*   **Workflow:**
    1.  Sender sends a message to the Request Queue.
    2.  Sender is free to perform other processing immediately after sending (asynchronous).
    3.  Sender performs a **blocking wait** on the Reply Queue when it requires the answer.
    4.  Receiver processes the request and sends the result to the Reply Queue.
    5.  Sender retrieves the response.

---

### Implementation Techniques

#### 1. Correlation IDs
This method is used when multiple responses sit in a shared Reply Queue. It ensures the sender retrieves only the response meant for its specific request.

*   **The Problem:** The Reply Queue may contain messages intended for other senders (e.g., IDs 120, 122).
*   **The Process:**
    1.  **Sender:** Sends a request with a unique **Message ID** (e.g., 124).
    2.  **Sender:** Waits on the Reply Queue using a **Message Selector** (or filter) looking for `CorrelationID == 124`.
    3.  **Receiver:** Gets the message, processes it, and sets the response's **Correlation ID** to match the original Message ID (124).
    4.  **Receiver:** Sends the message to the Reply Queue with a new unique Message ID (e.g., 857) but the matching Correlation ID.
    5.  **Sender:** Identifies the correct message via the Correlation ID and retrieves the data.

#### 2. Temporary Queues
A simpler alternative that does not use a shared reply queue initially.

*   **The Process:**
    1.  **Sender:** Sets a **"Reply To"** header in the message indicating a temporary queue (e.g., `TemporaryQueue T1`).
    2.  **Broker:** Creates this temporary queue; it is exclusive and unknown to others.
    3.  **Sender:** Performs a blocking wait on this specific temporary queue.
    4.  **Receiver:** Sends the response directly to the temporary queue specified in the header.
    5.  **Sender:** Receives the message (no selector/filter needed since the queue is private).
    6.  **Broker:** Removes the temporary queue once the interaction is complete.



## How Apache Kafka Differs From Standard Messaging

### Apache Kafka (Streaming Platform)
*   **Architecture Philosophy:** "Dumb Broker, Smart Consumer." The broker simply appends messages to a log; the consumer tracks its own position (offset).
*   **Data Type:** Good for **Operational Data** (metrics, logs, clickstreams, state changes).
*   **Payloads:** Optimized for small payloads (Key-Value pairs). Large payloads can degrade throughput significantly.
*   **Retention:** **Durable/Persistent by default.** Messages are retained for a configurable period (e.g., 7 days) or size, regardless of whether they have been consumed. This allows for "replayability."
*   **Throughput:** Extremely high (up to millions of messages/sec). Achieved through batching and sequential disk I/O.
*   **Topology:** Primarily **Publish/Subscribe** (Topics).
    *   Does not natively support complex routing logic (like message selectors or routing keys) inside the broker; this must be handled by the consumer or Kafka Streams.
*   **Scaling:** Horizontally scalable via **Partitioning**. Ordering is guaranteed only within a partition, not globally.

### Standard Messaging (RabbitMQ, ActiveMQ, JMS)
*   **Architecture Philosophy:** "Smart Broker, Dumb Consumer." The broker manages message state, delivery acknowledgments, and complex routing.
*   **Data Type:** Good for **Transactional Data** (orders, payments, user requests) where individual message guarantees are critical.
*   **Payloads:** Can handle larger payloads more gracefully than Kafka (though still limited by RAM/network).
*   **Retention:** **Transient by default.** Messages are typically deleted from the queue immediately after successful consumption (destructive read).
*   **Throughput:** Lower (approx. 4k–10k msgs/sec depending on persistence settings).
    *   *Note: While lower than Kafka, this is sufficient for most business transactional applications.*
*   **Topologies:** Supports complex, flexible topologies:
    *   **Point-to-Point (Queue):** Load balancing across consumers; message processed by only one consumer.
    *   **Publish/Subscribe (Topic):** Broadcast to all subscribers.
    *   **Request/Reply:** Natively supported via temporary queues and correlation IDs (as seen in Lesson 1).
*   **Routing:** sophisticated routing capabilities (e.g., Exchange types in RabbitMQ: Direct, Fanout, Topic, Headers) allow the broker to filter and route messages before they reach the consumer.

### Key Comparison Summary
| Feature | Kafka | Standard Messaging |
| :--- | :--- | :--- |
| **Message Lifecycle** | Log-based (persists until expiry) | Queue-based (deleted on consume) |
| **Consumer Complexity** | High (manages offsets) | Low (broker manages delivery) |
| **Ordering** | Guaranteed per partition | Guaranteed per queue (usually) |
| **Ideal Use Case** | Stream processing, Event Sourcing, Logging | Task queues, Complex routing, Request/Reply |

# Soft Skills

## The Knowledge Pyramid: Developers vs. Architects

Continously analyze technology and industry trends and keep current with the latest trends.

**The Three Tiers of Knowledge**
*   **Top:** Things You Know (Requires constant time investment to maintain).
*   **Middle:** Things You Know You Don’t Know.
*   **Bottom:** Things You Don’t Know You Don’t Know.

![](https://nealford.com/images/mark-pyramid-pt1.png)

**Developer Focus: Technical Depth**
*   Early career focuses on expanding the **top tier** to build hands-on experience and expertise.
*   The size of the top tier represents a developer's **technical depth**.

**Architect Focus: Technical Breadth**

> Technical Breadth: Stuff you know (technical depth) + stuff you know you don't know.

*   **Breadth > Depth:** An architect's value lies in understanding a wide variety of technologies to solve problems, rather than having deep expertise in just one.
*   It is more beneficial to know five potential solutions to a problem than to be a singular expert in one.
*   **Key Strategy:** Architects must sacrifice maintaining some of their deep, hard-won expertise and use that time to expand their **middle tier**, effectively broadening their technological portfolio to better match capabilities to constraints.


Few places you can expand your breadth knowledge
- InfoQ
- ThoughtWorks - Technology Rader
- DZone

---

**20 Minutes rule**

- Every morning, at the start of the day focus on expanding this area